In [1]:
pip install ultralytics opencv-python deep-sort-realtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 53.3 MB/s eta 0:00:00


In [2]:
import cv2
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [26]:
#input_video = r"sample_data/person-bicycle-car-detection.mp4"
#input_video = r"sample_data/one-by-one-person-detection.mp4"
input_video = r"sample_data/sample.mp4.mp4"
output_video =r"sample_data/output_tracked.mp4"

In [10]:
# -----------------------------
# 2) Load latest YOLO model
#    Examples:
#    yolo11n.pt, yolo11s.pt, yolo11m.pt
# -----------------------------
model = YOLO("yolo11s.pt")

In [17]:
# -----------------------------
# 3) Initialize Deep SORT
# -----------------------------
tracker = DeepSort(
    max_age=10,
    n_init=2,
    nms_max_overlap=0.7,
    max_cosine_distance=0.2,
    nn_budget=100,
    embedder="mobilenet",
    half=True,
    bgr=True,
    embedder_gpu=True,
)

In [24]:
# -----------------------------
# 4) Open video
# -----------------------------
cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    raise ValueError(f"Could not open video: {input_video}")


frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 25

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (frame_width, frame_height),
)

# COCO class names from model
class_names = model.names

In [27]:
# -----------------------------
# 5) Process video frame by frame
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO inference
    results = model.predict(
        source=frame,
        conf=0.25,
        iou=0.45,
        verbose=False
    )

    result = results[0]

    # Prepare detections for Deep SORT
    # Format expected:
    # [([left, top, width, height], confidence, class_name), ...]
    detections_for_tracker = []

    if result.boxes is not None and len(result.boxes) > 0:
        boxes_xyxy = result.boxes.xyxy.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)

        allowed_classes = {"person", "car", "bicycle"}

        for box, conf, cls_id in zip(boxes_xyxy, confidences, classes):
            x1, y1, x2, y2 = box
            w = x2 - x1
            h = y2 - y1

            class_name = class_names[cls_id]

            if class_name not in allowed_classes:
              continue

            detections_for_tracker.append(
                ([float(x1), float(y1), float(w), float(h)], float(conf), class_name)
            )

    # Update tracker
    tracks = tracker.update_tracks(detections_for_tracker, frame=frame)

    # Draw tracked objects
    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        ltrb = track.to_ltrb()  # left, top, right, bottom
        x1, y1, x2, y2 = map(int, ltrb)

        track_class = track.get_det_class()
        if track_class is None:
            track_class = "object"

        label = f"ID {track_id} | {track_class}"

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            frame,
            label,
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )

    writer.write(frame)
    #cv2.imshow("YOLO + Deep SORT Multi-Object Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
writer.release()
cv2.destroyAllWindows()
print(f"Done. Saved output to: {output_video}")

Done. Saved output to: sample_data/output_tracked.mp4
